## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-18-02-50-00 +0000,wsj,Singapore Dollar Weakens Slightly Amid Risk-Of...,https://www.wsj.com/finance/currencies/asian-c...
1,2026-11-18-02-47-43 +0000,nyt,Live Updates: Trump to Meet Philippine Leader ...,https://www.nytimes.com/live/2026/11/17/world/...
2,2026-11-18-02-47-41 +0000,nyt,Here’s the latest.,https://www.nytimes.com/live/2026/11/17/world/...
3,2026-11-18-02-44-52 +0000,nypost,NY school boss accused of possessing child por...,https://nypost.com/2026/11/17/us-news/ny-schoo...
4,2026-11-18-02-40-45 +0000,bbc,Jeff Bezos says he was 'heavily involved' in c...,https://www.bbc.co.uk/news/articles/cm5y1eejk2...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2747/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
10,trump,61
128,nvidia,36
244,huang,26
40,new,25
111,will,21


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
95,2026-11-17-23-02-12 +0000,bbc,"As Trump and Xi meet, China's AI dreams face a...",https://www.bbc.co.uk/news/articles/c5vg23g92r...,162
371,2026-11-17-12-57-00 +0000,wsj,Nvidia formed a board committee to oversee nat...,https://www.wsj.com/politics/national-security...,160
249,2026-11-17-18-34-00 +0000,wsj,A lawyer for Fed governor Lisa Cook said she w...,https://www.wsj.com/politics/policy/feds-lisa-...,158
255,2026-11-17-18-24-21 +0000,bbc,Jensen Huang says he would be 'honoured' to re...,https://www.bbc.co.uk/news/articles/cwx2en2vrx...,141
98,2026-11-17-22-57-29 +0000,nypost,Nvidia board chairman shoots down rumors disgr...,https://nypost.com/2026/11/17/business/nvidia-...,138


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
95,162,2026-11-17-23-02-12 +0000,bbc,"As Trump and Xi meet, China's AI dreams face a...",https://www.bbc.co.uk/news/articles/c5vg23g92r...
371,119,2026-11-17-12-57-00 +0000,wsj,Nvidia formed a board committee to oversee nat...,https://www.wsj.com/politics/national-security...
249,97,2026-11-17-18-34-00 +0000,wsj,A lawyer for Fed governor Lisa Cook said she w...,https://www.wsj.com/politics/policy/feds-lisa-...
211,84,2026-11-17-20-01-00 +0000,wsj,Progressive activists say the Senate next year...,https://www.wsj.com/politics/policy/the-angry-...
295,63,2026-11-17-17-00-56 +0000,chitri,Thanksgiving travel plans? AAA predicts fewer ...,https://www.chicagotribune.com/2026/11/17/than...
201,62,2026-11-17-20-31-11 +0000,nyt,Ukraine Talks Run Up Against Deep Enmity Betwe...,https://www.nytimes.com/2026/11/17/us/politics...
120,60,2026-11-17-22-33-00 +0000,wsj,"Hung Cao, President Trump’s pick to lead the N...",https://www.wsj.com/politics/national-security...
369,53,2026-11-17-13-07-05 +0000,cbc,Iran and U.S. trade fire as Iran denies report...,https://www.cbc.ca/news/world/iran-us-war-bahr...
304,52,2026-11-17-16-40-51 +0000,nypost,DHS boss Markwayne Mullin defends flying to co...,https://nypost.com/2026/11/17/us-news/markwayn...
328,50,2026-11-17-15-35-04 +0000,nypost,Clarence Thomas calls foul on California rulin...,https://nypost.com/2026/11/17/us-news/clarence...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
